# Black Box Courier / Antinode Courier — Manim Presentation Notebook

**Pi 30 Innovation Sprint project notebook**  
This notebook explains the **Courier game only** from the uploaded ESP32 console build. It focuses on the game concept, player loop, scoring, RF-interference idea, possible logic issues, and how to present it using **Manim** animations.

Core source analysed: `GameLogicBlackBoxCourier.h`  
Game modes covered: **Antinode Run / Black Box Hunter** and **Dead Air Mode**.  
Not covered: Bluetooth tools, Wi-Fi tools, mouse app, launcher utilities, or any non-game audit pages.

## Manim 0.20.1 compatibility fix

This version replaces `LaggedStartMap(Create, arrows, ...)` with `LaggedStart(*[GrowArrow(a) for a in arrows], ...)` in the arrow-heavy scenes. In Manim Community v0.20.1, `Create` can fail on `Arrow` because an arrow includes an internal `ArrowTriangleFilledTip` submobject.

## 0. How to run this notebook

Run this in a Jupyter environment where Manim is installed. If Manim is missing, run the setup cell below first. On Raspberry Pi, rendering can be slow, so use `-ql` or `-qm` quality for practice.

Recommended presentation flow:
1. Run each Manim cell once before presenting.
2. Keep the rendered video outputs visible under each cell.
3. Use the markdown explanations as your speaker notes.

In [ ]:
# Setup cell. Run once if Manim is not installed in your notebook environment.
# If you already have Manim, you can skip this.
# On Linux/Raspberry Pi, also install system dependencies if needed:
# sudo apt install ffmpeg libcairo2-dev libpango1.0-dev texlive texlive-latex-extra

# %pip install manim

In [1]:
from manim import *
import numpy as np

config.media_width = "90%"
config.verbosity = "WARNING"

## 1. One-line game idea

**Black Box Courier** is a small OLED RF puzzle game where the player must locate a hidden courier beacon in the 2.4 GHz/nRF channel space, tune the receiver, sample at good interference timing, collect fragments or packets, and verify the delivery.

The important design choice is that **FIRE is not an instant win button**. FIRE costs battery/score/trace, has cooldown, and only works well when the player has built a good lock using:

- correct RF lane/channel,
- correct data rate,
- correct power level when the level requires it,
- low Wi-Fi/BLE/ghost risk,
- good antinode timing,
- suitable FAST/MED/SLOW filter.

In [2]:
%%manim -qm -v WARNING GamePitchScene

from manim import *
import numpy as np

class GamePitchScene(Scene):
    def construct(self):
        title = Text("Black Box Courier", font_size=46, weight=BOLD).to_edge(UP)
        subtitle = Text("An RF puzzle game: scan → tune → sample → decode", font_size=25).next_to(title, DOWN)
        self.play(Write(title), FadeIn(subtitle, shift=DOWN))

        oled = RoundedRectangle(width=5.2, height=3.1, corner_radius=0.18).shift(LEFT*3.0 + DOWN*0.2)
        oled_title = Text("128×64 OLED", font_size=22).move_to(oled.get_top()+DOWN*0.25)
        rf_bars = VGroup()
        for i, h in enumerate([0.4, 0.9, 0.6, 1.7, 2.2, 1.1, 1.8, 0.5]):
            bar = Rectangle(width=0.28, height=h).set_fill(WHITE, opacity=1).set_stroke(WHITE, 1)
            bar.move_to(oled.get_bottom()+UP*(0.55+h/2)+LEFT*1.9+RIGHT*i*0.55)
            rf_bars.add(bar)
        beacon = Dot(rf_bars[4].get_top()+UP*0.2).set_color(YELLOW)
        label = Text("hidden beacon", font_size=18).next_to(beacon, UP)

        player = VGroup(
            RoundedRectangle(width=4.0, height=2.6, corner_radius=0.2),
            Text("Player choices", font_size=26).shift(UP*0.75),
            Text("CH • RATE • PWR • FILTER • TIMING", font_size=20).shift(DOWN*0.05),
            Text("FIRE = costly sample", font_size=20).shift(DOWN*0.75)
        ).shift(RIGHT*3.0 + DOWN*0.2)

        arrow = Arrow(oled.get_right()+RIGHT*0.25, player.get_left()+LEFT*0.25, buff=0.1)
        self.play(Create(oled), FadeIn(oled_title), LaggedStartMap(GrowFromEdge, rf_bars, edge=DOWN, lag_ratio=0.08))
        self.play(FadeIn(beacon), Write(label), GrowArrow(arrow), FadeIn(player))
        self.wait(1)

Manim Community v0.20.1

## 2. Game architecture from the source

The game class keeps the existing launcher interface by exposing `AntinodeGame`, but the main gameplay layer is the courier game.

Important structures:

| Part | Purpose |
|---|---|
| `GameState` | Menu, Play, Level Clear, Game Over, Help, Reset Confirm |
| `PlayMode` | Hunter/Courier run or Dead Air challenge |
| `UiScreen` | SCAN, TUNE, DECODE |
| `LevelPlan` | Defines difficulty: base channel, rate, symbol count, ghosts, hopping, Wi-Fi punishment, twin beacon |
| `BinInfo` | Per-lane RF/wifi/BLE/wave/phase/ghost/null information |
| `BeaconPacket` | Small nRF packet carrying level, symbol, channel, rate, sequence, checksum |

The game loop calls:

```cpp
readButtons(now)
updateClock(now)
updateBeaconPulse(now)
updateWifiScan(now)
handleInput(now)
render()
```

That means the game is state-driven: input changes state, periodic updates change the world, and rendering visualizes the current state.

In [3]:
%%manim -qm -v WARNING ArchitectureScene

from manim import *

class ArchitectureScene(Scene):
    def box(self, text, pos, color=WHITE):
        r = RoundedRectangle(width=3.0, height=0.75, corner_radius=0.14).set_stroke(color, 2)
        t = Text(text, font_size=22).move_to(r)
        return VGroup(r, t).move_to(pos)

    def construct(self):
        title = Text("Game architecture", font_size=42, weight=BOLD).to_edge(UP)
        self.play(Write(title))
        boxes = [
            self.box("Buttons", LEFT*4 + UP*1.7),
            self.box("Game State", LEFT*1.25 + UP*1.7, YELLOW),
            self.box("World Update", RIGHT*1.65 + UP*1.7),
            self.box("OLED Render", RIGHT*4.25 + UP*1.7),
            self.box("SCAN screen", LEFT*3.0 + DOWN*0.6),
            self.box("TUNE screen", ORIGIN + DOWN*0.6),
            self.box("DECODE screen", RIGHT*3.0 + DOWN*0.6),
            self.box("Score / Battery / Trace", DOWN*2.3, BLUE),
        ]
        arrows = VGroup(
            Arrow(boxes[0].get_right(), boxes[1].get_left(), buff=0.1),
            Arrow(boxes[1].get_right(), boxes[2].get_left(), buff=0.1),
            Arrow(boxes[2].get_right(), boxes[3].get_left(), buff=0.1),
            Arrow(boxes[1].get_bottom(), boxes[5].get_top(), buff=0.15),
            Arrow(boxes[4].get_right(), boxes[5].get_left(), buff=0.1),
            Arrow(boxes[5].get_right(), boxes[6].get_left(), buff=0.1),
            Arrow(boxes[5].get_bottom(), boxes[7].get_top(), buff=0.1),
        ).set_stroke(width=3)
        # Manim 0.20.1 can mis-handle LaggedStartMap(Create, arrows) because
        # Arrow contains an ArrowTriangleFilledTip submobject. GrowArrow is safer.
        self.play(
            LaggedStart(*[FadeIn(b) for b in boxes], lag_ratio=0.08),
            LaggedStart(*[GrowArrow(a) for a in arrows], lag_ratio=0.08),
        )
        note = Text("Every frame: read input → update game variables → render OLED", font_size=24).to_edge(DOWN)
        self.play(FadeIn(note, shift=UP))
        self.wait(1)

Manim Community v0.20.1

## 3. Player loop: why it is a proper game

A proper game needs meaningful player choices, consequences, feedback, and failure states. This courier game has those:

1. **SCAN**: inspect RF lanes, Wi-Fi/BLE risk, ghost/null clues.
2. **TUNE**: select channel, data rate, power, and filter.
3. **SAMPLE**: press FIRE to spend battery/score/trace and attempt capture.
4. **DECODE**: verify collected fragments or packets.
5. **Progression**: later levels introduce rate matching, power matching, hopping beacon, Wi-Fi punishment, BLE decoys, ghost lanes, and twin beacon.

So the core is valid: the player cannot reliably win by random FIRE alone, because bad samples reduce score, battery, confidence, and increase trace.

In [4]:
%%manim -qm -v WARNING PlayerLoopScene

from manim import *

class PlayerLoopScene(Scene):
    def node(self, name, pos, color):
        c = Circle(radius=0.78).set_stroke(color, 4).move_to(pos)
        t = Text(name, font_size=24, weight=BOLD).move_to(c)
        return VGroup(c, t)

    def construct(self):
        title = Text("Core gameplay loop", font_size=42, weight=BOLD).to_edge(UP)
        self.play(Write(title))
        scan = self.node("SCAN", LEFT*3.8 + UP*0.8, BLUE)
        tune = self.node("TUNE", ORIGIN + UP*0.8, YELLOW)
        sample = self.node("FIRE", RIGHT*3.8 + UP*0.8, RED)
        decode = self.node("DECODE", ORIGIN + DOWN*1.5, GREEN)
        arrows = VGroup(
            Arrow(scan.get_right(), tune.get_left(), buff=0.1),
            Arrow(tune.get_right(), sample.get_left(), buff=0.1),
            Arrow(sample.get_bottom(), decode.get_right(), buff=0.1),
            Arrow(decode.get_left(), scan.get_bottom(), buff=0.1),
        )
        # Use GrowArrow instead of LaggedStartMap(Create, arrows) for Manim 0.20.1.
        self.play(
            FadeIn(scan), FadeIn(tune), FadeIn(sample), FadeIn(decode),
            LaggedStart(*[GrowArrow(a) for a in arrows], lag_ratio=0.12),
        )
        costs = VGroup(
            Text("FIRE costs:", font_size=26, weight=BOLD),
            Text("battery ↓", font_size=22),
            Text("score ↓ on fail", font_size=22),
            Text("trace ↑", font_size=22),
            Text("cooldown", font_size=22),
        ).arrange(DOWN, aligned_edge=LEFT).to_edge(RIGHT).shift(DOWN*1.0)
        self.play(FadeIn(costs, shift=LEFT))
        self.wait(1)

Manim Community v0.20.1

## 4. SCAN screen explained

The SCAN screen compresses the 126 nRF24 channels into 8 RF lanes. Each lane stores:

- `rf`: energy/packet evidence,
- `wifi`: Wi-Fi pressure,
- `ble`: simulated BLE/decoy pressure,
- `wave`: interference strength,
- `phase`: timing clue,
- `peakCh`: best channel inside the lane,
- `ghost`: fake peak/decoy flag,
- `nullField`: destructive-interference warning.

The player chooses a lane with left/right. Pressing DOWN tunes to the best channel in the lane. This is good design because the map is not only decoration; it directly drives tuning.

In [5]:
%%manim -qm -v WARNING ScanScreenScene

from manim import *
import numpy as np

class ScanScreenScene(Scene):
    def construct(self):
        title = Text("SCAN screen: 8 RF lanes", font_size=40, weight=BOLD).to_edge(UP)
        self.play(Write(title))
        oled = RoundedRectangle(width=10.5, height=4.5, corner_radius=0.18).shift(DOWN*0.3)
        header = Text("RF MAP   L03   B94%   T1", font_size=24).move_to(oled.get_top()+DOWN*0.35)
        self.play(Create(oled), FadeIn(header))
        heights = [1.0, 1.6, 1.2, 2.6, 1.4, 3.0, 1.9, 0.8]
        labels = ["0","1","2","3","4","5","6","7"]
        bars = VGroup()
        for i, h in enumerate(heights):
            x = -4.0 + i*1.15
            frame = Rectangle(width=0.65, height=2.8).move_to([x, -0.25, 0]).set_stroke(WHITE, 1)
            bar = Rectangle(width=0.55, height=h).set_fill(WHITE, opacity=0.9).set_stroke(WHITE, 0).align_to(frame, DOWN).move_to([x, -1.65 + h/2, 0])
            num = Text(labels[i], font_size=18).next_to(frame, DOWN, buff=0.1)
            bars.add(VGroup(frame, bar, num))
        focus = SurroundingRectangle(bars[5], buff=0.08).set_stroke(YELLOW, 3)
        ghost = Text("GHOST", font_size=18, color=RED).next_to(bars[3], UP)
        anti = Text("ANTINODE", font_size=18, color=YELLOW).next_to(bars[5], UP)
        null = Text("NULL", font_size=18, color=BLUE).next_to(bars[1], UP)
        bottom = Text("LANE:5  BEST:5  RISK:L", font_size=24).move_to(oled.get_bottom()+UP*0.35)
        self.play(LaggedStartMap(FadeIn, bars, lag_ratio=0.06), Create(focus), FadeIn(ghost), FadeIn(anti), FadeIn(null), FadeIn(bottom))
        self.wait(1)

Manim Community v0.20.1

## 5. TUNE screen explained

The TUNE screen asks for four real decisions:

| Setting | Why it matters |
|---|---|
| `CH` | Actual nRF channel must match the hidden beacon or at least the correct lane. |
| `RATE` | Later levels require the receiver data rate to match the beacon rate. |
| `PWR` | Later levels punish wrong transmit/listen strength. Too much power also increases trace. |
| `FILT` | FAST/MED/SLOW filtering converts interference into a skill choice. |

The lock score combines channel distance, rate match, power match, pulse timing, wave quality, filter quality, risks, ghost penalty, and trace penalty.

In [6]:
%%manim -qm -v WARNING TuneScoreScene

from manim import *

class TuneScoreScene(Scene):
    def construct(self):
        title = Text("TUNE screen: lock is earned", font_size=40, weight=BOLD).to_edge(UP)
        self.play(Write(title))
        left = VGroup(
            Text("> CH   076", font_size=28),
            Text("  RATE 1M", font_size=28),
            Text("  PWR  LOW", font_size=28),
            Text("  FILT MED", font_size=28),
            Text("F = SAMPLE", font_size=24),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.25).shift(LEFT*4.2 + DOWN*0.2)
        self.play(FadeIn(left, shift=RIGHT))
        factors = [
            ("channel", 40), ("rate", 22), ("power", 16),
            ("timing", 12), ("wave", 18), ("filter", 8), ("risk penalties", -20)
        ]
        rows = VGroup()
        y = 1.6
        for name, val in factors:
            text = Text(f"{name:>13}: {val:+}", font_size=22)
            text.move_to(RIGHT*2.2 + UP*y)
            rows.add(text)
            y -= 0.42
        self.play(LaggedStartMap(FadeIn, rows, lag_ratio=0.08))
        meter_frame = Rectangle(width=4.8, height=0.42).shift(RIGHT*2.2 + DOWN*1.8)
        meter_fill = Rectangle(width=3.75, height=0.32).set_fill(YELLOW, opacity=0.9).set_stroke(YELLOW, 0).align_to(meter_frame, LEFT).move_to(meter_frame.get_left()+RIGHT*1.875)
        label = Text("LOCK 78/100: sample window is good", font_size=24).next_to(meter_frame, DOWN)
        self.play(Create(meter_frame), GrowFromEdge(meter_fill, LEFT), FadeIn(label))
        self.wait(1)

Manim Community v0.20.1

## 6. Interference field: the science concept

The game already models a simplified multipath field:

\[
A(x,t)=\left|D(x,t)+R(x,t)+G(x,t)\right|
\]

Where:

- `D` is the direct beacon wave,
- `R` is reflected multipath,
- `G` is decoy/twin/ghost contribution,
- antinode = constructive addition,
- null = destructive cancellation.

This is the strongest innovation point. The game is not just “find the highest bar”; later levels ask the player to read the wave phase and choose when to sample.

In [7]:
%%manim -qm -v WARNING InterferenceScene

from manim import *
import numpy as np

class InterferenceScene(Scene):
    def construct(self):
        title = Text("Interference field: antinode vs null", font_size=39, weight=BOLD).to_edge(UP)
        self.play(Write(title))
        axes = Axes(x_range=[0, 8, 1], y_range=[-2, 2, 1], x_length=10, y_length=3.5, tips=False).shift(DOWN*0.4)
        x_label = Text("RF lane", font_size=20).next_to(axes.x_axis, DOWN)
        self.play(Create(axes), FadeIn(x_label))
        wave1 = axes.plot(lambda x: np.cos(2.2*x), x_range=[0,8], use_smoothing=True)
        wave2 = axes.plot(lambda x: 0.7*np.cos(2.2*x - 1.1), x_range=[0,8], use_smoothing=True)
        total = axes.plot(lambda x: np.cos(2.2*x)+0.7*np.cos(2.2*x - 1.1), x_range=[0,8], use_smoothing=True).set_stroke(width=6)
        lab1 = Text("direct", font_size=18).set_color(wave1.get_color()).to_corner(LEFT+DOWN)
        lab2 = Text("reflected", font_size=18).set_color(wave2.get_color()).next_to(lab1, RIGHT, buff=0.5)
        lab3 = Text("combined field", font_size=18).next_to(lab2, RIGHT, buff=0.5)
        self.play(Create(wave1), FadeIn(lab1))
        self.play(Create(wave2), FadeIn(lab2))
        self.play(Create(total), FadeIn(lab3))
        anti_x = 2.85
        null_x = 5.0
        anti = Dot(axes.c2p(anti_x, np.cos(2.2*anti_x)+0.7*np.cos(2.2*anti_x-1.1)), color=YELLOW).scale(1.6)
        null = Dot(axes.c2p(null_x, np.cos(2.2*null_x)+0.7*np.cos(2.2*null_x-1.1)), color=BLUE).scale(1.6)
        anti_label = Text("ANTINODE\nfire window", font_size=22, color=YELLOW).next_to(anti, UP)
        null_label = Text("NULL\nwait", font_size=22, color=BLUE).next_to(null, DOWN)
        self.play(FadeIn(anti), FadeIn(anti_label), FadeIn(null), FadeIn(null_label))
        self.wait(1)

Manim Community v0.20.1

## 7. Filter choice: FAST / MED / SLOW

The source comments already connect this to Fourier/Laplace-style filtering. For a presentation, explain it like this:

- **FAST**: quick response, good for hopping and changing phase, but noisy.
- **MED**: balanced matched filter for normal levels.
- **SLOW**: suppresses Wi-Fi/BLE noise, but can lag behind hopping signals.

A more mathematically proper upgrade can use a Laplace-domain first-order low-pass filter:

\[
H(s)=\frac{1}{\tau s+1}
\]

Discrete update:

\[
y[n]=\alpha x[n]+(1-\alpha)y[n-1],\quad \alpha=\frac{\Delta t}{\tau+\Delta t}
\]

Small \(\tau\) = FAST. Large \(\tau\) = SLOW.

In [8]:
%%manim -qm -v WARNING FilterScene

from manim import *
import numpy as np

class FilterScene(Scene):
    def smooth(self, y, alpha):
        out = []
        prev = y[0]
        for v in y:
            prev = alpha*v + (1-alpha)*prev
            out.append(prev)
        return np.array(out)

    def construct(self):
        title = Text("Laplace/IIR filter idea for gameplay", font_size=38, weight=BOLD).to_edge(UP)
        self.play(Write(title))
        axes = Axes(x_range=[0, 10, 1], y_range=[-0.5, 2.5, 1], x_length=10, y_length=3.6, tips=False).shift(DOWN*0.4)
        self.play(Create(axes))
        xs = np.linspace(0, 10, 140)
        signal = 1.0 + 0.8*np.sin(3*xs) + 0.28*np.sin(17*xs)
        fast = self.smooth(signal, 0.55)
        med = self.smooth(signal, 0.18)
        slow = self.smooth(signal, 0.06)
        raw_curve = axes.plot_line_graph(xs, signal, add_vertex_dots=False).set_stroke(width=2)
        fast_curve = axes.plot_line_graph(xs, fast, add_vertex_dots=False).set_stroke(width=4)
        med_curve = axes.plot_line_graph(xs, med, add_vertex_dots=False).set_stroke(width=4)
        slow_curve = axes.plot_line_graph(xs, slow, add_vertex_dots=False).set_stroke(width=4)
        labels = VGroup(
            Text("raw RF", font_size=19),
            Text("FAST: tracks changes", font_size=19),
            Text("MED: balanced", font_size=19),
            Text("SLOW: noise suppression", font_size=19),
        ).arrange(DOWN, aligned_edge=LEFT).to_edge(RIGHT).shift(DOWN*0.2)
        self.play(Create(raw_curve), FadeIn(labels[0]))
        self.play(Create(fast_curve), FadeIn(labels[1]))
        self.play(Create(med_curve), FadeIn(labels[2]))
        self.play(Create(slow_curve), FadeIn(labels[3]))
        formula = MathTex(r"H(s)=\frac{1}{\tau s+1}", font_size=38).to_edge(DOWN)
        self.play(Write(formula))
        self.wait(1)

Manim Community v0.20.1

## 8. Decode and scoring

After enough successful samples, the player moves to DECODE and verifies the captured code.

Important consequences:

- successful capture increases confidence and score,
- failure increases errors and trace,
- low battery or too much trace ends the run,
- verification requires all fragments/packets,
- high score rewards clean play, correct timing, low power, antinode hits, and strong filters.

This means the game has a measurable skill curve instead of only random success.

In [9]:
%%manim -qm -v WARNING DecodeScoringScene

from manim import *

class DecodeScoringScene(Scene):
    def construct(self):
        title = Text("Decode: convert captures into a verified delivery", font_size=36, weight=BOLD).to_edge(UP)
        self.play(Write(title))
        slots = VGroup()
        for i, txt in enumerate(["A7", "3F", "C2", "__", "__"]):
            box = Square(side_length=0.85).set_stroke(WHITE, 2).shift(LEFT*2.0 + RIGHT*i*1.0 + UP*0.8)
            label = Text(txt, font_size=26).move_to(box)
            slots.add(VGroup(box, label))
        self.play(LaggedStartMap(FadeIn, slots, lag_ratio=0.1))
        score = VGroup(
            Text("Success", font_size=28, weight=BOLD),
            Text("confidence +2", font_size=22),
            Text("score +75 + bonuses", font_size=22),
            Text("may hop beacon", font_size=22),
        ).arrange(DOWN, aligned_edge=LEFT).shift(LEFT*3.2 + DOWN*1.1)
        fail = VGroup(
            Text("Failure", font_size=28, weight=BOLD),
            Text("errors +1", font_size=22),
            Text("trace +1", font_size=22),
            Text("score −50", font_size=22),
        ).arrange(DOWN, aligned_edge=LEFT).shift(RIGHT*2.5 + DOWN*1.1)
        self.play(FadeIn(score, shift=UP), FadeIn(fail, shift=UP))
        verify = Text("VERIFY only when all fragments are captured", font_size=28, color=YELLOW).to_edge(DOWN)
        self.play(FadeIn(verify))
        self.wait(1)

Manim Community v0.20.1

## 9. Technical judgment: is it a valid playable game?

**Yes — it can be called a playable game.** It has goals, progression, feedback, player decisions, risk/reward, failure states, scoring, and escalating mechanics. It is not just “press random FIRE and win.”

However, these are the important issues to discuss before a final sprint demo:

| Issue | Why it matters | Suggested fix |
|---|---|---|
| Random fallback can still grant captures if the lock is “likely” | Demo-friendly, but may feel probabilistic rather than purely skill-based | Show exact lock probability or replace with deterministic threshold for competition mode |
| Battery resets every level | Makes each level standalone, but weakens long-run resource pressure | Add “campaign mode” where battery carries across levels |
| `updateBeaconPulse()` changes RF bars between scans | Makes bars feel alive, but may temporarily overwrite null/ghost visual detail | Only animate visual overlay, not the stored bin state used for decisions |
| Ghost lane can still have high energy | Good for deception, but players need fair clues | Make ghost marker/phase mismatch teachable by level 2/3 tutorial |
| XOR checksum is not strong | Fine for game packets, not robust validation | Use CRC-8/CRC-16 if packet correctness matters |
| FIRE release handling can accidentally sample after a near-hold | Usability issue | Add a dead-zone: 750–1100 ms release does no action or shows “hold more” |
| Real RF hardware may be noisy or inconsistent | Classroom demo risk | Keep SIM mode available and display `RF OK`/`SIM` clearly |

**Overall gameplay rating for Courier game:** **82/100**  
It is valid and presentation-worthy. The biggest improvement is to make the interference/filter mechanic more explicit and more deterministic, so the player learns a real skill rather than feeling that sampling is chance-based.

In [10]:
%%manim -qm -v WARNING ProblemFixScene

from manim import *

class ProblemFixScene(Scene):
    def construct(self):
        title = Text("Main demo risks and fixes", font_size=40, weight=BOLD).to_edge(UP)
        self.play(Write(title))
        items = [
            ("Random fallback", "show probability / competition threshold"),
            ("Battery resets", "campaign battery carry-over"),
            ("Animated bins mutate logic", "visual overlay only"),
            ("Ghosts can feel unfair", "teach phase mismatch clue"),
            ("Near-hold FIRE", "add no-action dead zone"),
        ]
        group = VGroup()
        for left, right in items:
            l = Text(left, font_size=22).set_x(-3.2)
            r = Text(right, font_size=22).set_x(2.2)
            arrow = Arrow(l.get_right()+RIGHT*0.15, r.get_left()+LEFT*0.15, buff=0.05)
            row = VGroup(l, arrow, r)
            group.add(row)
        group.arrange(DOWN, buff=0.45, aligned_edge=LEFT).shift(DOWN*0.2)
        self.play(LaggedStartMap(FadeIn, group, lag_ratio=0.1))
        rating = Text("Courier game rating: 82 / 100", font_size=34, weight=BOLD, color=YELLOW).to_edge(DOWN)
        self.play(FadeIn(rating, shift=UP))
        self.wait(1)

Manim Community v0.20.1

## 10. Better real-interference upgrade for the project

To make the project more “proper game + real signal concept,” use a deterministic RF skill model:

### A. Keep a live energy history per lane
Store recent samples:

\[
x_i[n] = \text{RF energy of lane } i \text{ at time } n
\]

### B. Apply three Laplace/IIR filters

\[
y_{fast}, y_{med}, y_{slow}
\]

Each filter uses a different \(\tau\). The selected filter changes the displayed lock and capture score.

### C. Estimate antinode timing
Compute a normalized phase/gradient metric:

\[
q_i[n] = \frac{|y_i[n]-y_i[n-1]|}{\epsilon + \max(y_i)}
\]

Then reward a player who samples near a stable high-energy antinode, not just any spike.

### D. Replace pure random success with transparent probability

\[
P(capture)=\sigma(w_c C + w_f F + w_a A - w_n N - w_t T)
\]

Where:
- `C` = channel/rate/power correctness,
- `F` = filter match,
- `A` = antinode quality,
- `N` = noise/ghost/null risk,
- `T` = trace penalty.

For competitive mode, use `capture = P > 0.78` instead of random draw.

In [11]:
%%manim -qm -v WARNING FinalConceptScene

from manim import *

class FinalConceptScene(Scene):
    def construct(self):
        title = Text("Final presentation message", font_size=42, weight=BOLD).to_edge(UP)
        self.play(Write(title))
        msg = VGroup(
            Text("Not a button-masher.", font_size=34, weight=BOLD),
            Text("It is a small RF decision game.", font_size=30),
            Text("The player reads lanes, phase, ghosts, nulls, and filters.", font_size=25),
            Text("Manim visualizes the hidden science behind the OLED game.", font_size=25),
        ).arrange(DOWN, buff=0.35).shift(DOWN*0.2)
        self.play(LaggedStartMap(FadeIn, msg, lag_ratio=0.25))
        footer = Text("Pi 30 Innovation Sprint: game + interference learning", font_size=24, color=YELLOW).to_edge(DOWN)
        self.play(FadeIn(footer))
        self.wait(1.5)

Manim Community v0.20.1

## 11. Optional: export all scenes from terminal

If you prefer rendering outside Jupyter, save each Manim scene into a Python file and run:

```bash
manim -qm courier_presentation.py GamePitchScene
manim -qm courier_presentation.py ArchitectureScene
manim -qm courier_presentation.py PlayerLoopScene
manim -qm courier_presentation.py ScanScreenScene
manim -qm courier_presentation.py TuneScoreScene
manim -qm courier_presentation.py InterferenceScene
manim -qm courier_presentation.py FilterScene
manim -qm courier_presentation.py DecodeScoringScene
manim -qm courier_presentation.py ProblemFixScene
manim -qm courier_presentation.py FinalConceptScene
```

For Raspberry Pi practice, use `-ql` for faster low-quality renders.